In [2]:
import os
import regex as re
from typing import BinaryIO

In [ ]:
def get_token_frequencies(text, pattern, special_tokens=None):
    """
    Takes text and a compiled regex pattern.
    Returns a frequency table: { 'token': count, ... }
    """
    stats = {}
    
    # Step 1: Split on special tokens if provided
    if special_tokens:
        # Build pattern: escape each special token and join with |
        split_pattern = "|".join(re.escape(token) for token in special_tokens)
        # Split the text, keeping only non-empty chunks
        chunks = [chunk for chunk in re.split(split_pattern, text) if chunk]
    else:
        chunks = [text]
    
    # Step 2: Pre-tokenize each chunk separately
    for chunk in chunks:
        # iterate using finditer (memory efficient)
        for match in pattern.finditer(chunk):
            # Extract the actual string matched
            token = match.group()
            
            # Manual counting logic
            if token in stats:
                stats[token] += 1
            else:
                stats[token] = 1
            
    return stats


def get_pair_frequencies(vocab):
    """
    Given a vocab dict like {('l','o','w'): 5, ('l','o','w','e','r'): 2}
    Returns pair counts like {('l','o'): 7, ('o','w'): 7, ('w','e'): 2, ...}
    """
    pairs = {}
    
    # Loop through each token and its frequency
    for token, freq in vocab.items():
        # Look at consecutive pairs in this token
        # Example: ('l','o','w') → pairs are ('l','o') and ('o','w')
        for i in range(len(token) - 1):
            # Get pair at position i
            pair = (token[i], token[i + 1])
            
            # Add this pair's count (weighted by token frequency)
            if pair in pairs:
                pairs[pair] += freq
            else:
                pairs[pair] = freq
    
    return pairs


def get_best_pair(pair_frequencies):
    """
    Returns the pair with highest frequency.
    In case of tie, returns lexicographically greater pair.
    """
    if not pair_frequencies:
        return None
    
    # Find the maximum frequency
    max_freq = max(pair_frequencies.values())
    
    # Get all pairs with that frequency
    top_pairs = [pair for pair, freq in pair_frequencies.items() if freq == max_freq]
    
    # Return the lexicographically greatest one
    # max() on tuples compares element by element
    return max(top_pairs)


def merge_pair(vocab, pair_to_merge):
    """
    Takes vocab and a pair like ('s', 't')
    Returns new vocab where that pair is merged everywhere
    Example: ('n','e','w','e','s','t') → ('n','e','w','e','st') if merging ('s','t')
    """
    new_vocab = {}
    
    for token, freq in vocab.items():
        # We'll build a new version of this token
        new_token = []
        i = 0
        
        while i < len(token):
            # Check if we can merge at position i
            if i < len(token) - 1 and (token[i], token[i + 1]) == pair_to_merge:
                # Merge! Combine the two characters
                merged_char = token[i] + token[i + 1]
                new_token.append(merged_char)
                i += 2  # Skip both characters
            else:
                # Don't merge, just copy
                new_token.append(token[i])
                i += 1
        
        # Add to new vocab
        new_vocab[tuple(new_token)] = freq
    
    return new_vocab


def train_bpe(input_path="", vocab_size=1000, special_tokens=["<|endoftext|>"]):

    with open(input_path, "rb") as f:
        raw_data = f.read(vocab_size).decode("utf-8", errors="ignore")
            # Run pre-tokenization on your chunk and store the counts for each pre-token
        print(raw_data)

    compiled_pattern = re.compile(PAT)
    token_frequencies = get_token_frequencies(raw_data, compiled_pattern, special_tokens)
    vocab = {tuple(list(token)): freq for token, freq in token_frequencies.items()}
    
    merges = []  # Track the sequence of merges
    
    for merge_num in range(num_merges):
        # Step 1: Count all pairs
        pair_freq = get_pair_frequencies(vocab)
        
        if not pair_freq:
            print(f"No more pairs to merge at step {merge_num}")
            break
        
        # Step 2: Find best pair
        best_pair = get_best_pair(pair_freq)
        
        # Step 3: Merge it
        vocab = merge_pair(vocab, best_pair)
        
        # Step 4: Record this merge
        merge_str = f"{best_pair[0]} {best_pair[1]}"
        merges.append(merge_str)
        
        print(f"Merge {merge_num + 1}: '{merge_str}' (freq: {pair_freq[best_pair]})")
        
    return vocab, merges

vocab, merges = train_bpe(input_path = "../data/TinyStoriesV2-GPT4-train.txt", vocab_size=1000, special_tokens=["<|endoftext|>"])
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""


print("\n" + "="*50)
print("Final merges:", merges)
print("\nFinal vocabulary:")
for token, freq in vocab.items():
    print(f"{''.join(token)}: {freq}")


Once upon a time there was a little boy named Ben. Ben loved to explore the world around him. He saw many amazing things, like beautiful vases that were on display in a store. One day, Ben was walking through the store when he came across a very special vase. When Ben saw it he was amazed!  
He said, “Wow, that is a really amazing vase! Can I buy it?” 
The shopkeeper smiled and said, “Of course you can. You can take it home and show all your friends how amazing it is!”
So Ben took the vase home and he was so proud of it! He called his friends over and showed them the amazing vase. All his friends thought the vase was beautiful and couldn't believe how lucky Ben was. 
And that's how Ben found an amazing vase in the store!
<|endoftext|>
Once upon a time, there was a reliable otter named Ollie. He lived in a river with his family. They all loved to play and swim together.
One day, Ollie's mom said, "Ollie, hurry and get some fish for dinner!" Ollie swam fast to catch fish. He saw


NameError: name 'compiled_pattern' is not defined